# 01 - Preparação de Dados

Carrega o arquivo `espelhos_acordaos_artigo2026.parquet` e complementa com o **texto integral** de cada decisão, obtido dos ZIPs publicados na Base de Dados Abertos do STJ (dataset `integras-de-decisoes-terminativas-e-acordaos-do-diario-da-justica`).
- Dados Abertos: https://dadosabertos.web.stj.jus.br/group/jurisprudencia
- Sobre o espelho do acórdão: https://scon.stj.jus.br/docs_internet/jurisprudencia/ajuda/Espelho_do_Acordao_atualizado.pdf
- Sobre a api CKAN: https://docs.ckan.org/en/2.9/api/

No caso de have problemas com o download de algum arquivo usando a api:
- Download manual dos arquivos com as íntegras: https://dadosabertos.web.stj.jus.br/dataset/integras-de-decisoes-terminativas-e-acordaos-do-diario-da-justica
- Download manual dos arquivos com os espelhos: https://dadosabertos.web.stj.jus.br/organization/superior-tribunal-de-justica?tags=jurisprudencia

## Estratégia otimizada
Em vez de fazer uma requisição à API do CKAN por documento (lento),
a abordagem baixa cada ZIP **uma única vez** e varre seus membros em memória,
extraindo apenas os TXTs cujos `seq_documento_acordao` estão nos registros do parquet.

```
parquet → set(seq_documento_acordao)
         ↓
CKAN → lista de ZIPs
         ↓  (download streaming)
ZIP em memória → filtro por seq_documento → texto
         ↓
parquet enriquecido com coluna 'texto'
```

## 1. Configuração e Imports

In [1]:
import os
import io
import zipfile
import requests
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm

# ─── CKAN ─────────────────────────────────────────────────────────────────────
CKAN_BASE_URL = 'https://dadosabertos.web.stj.jus.br'

# Dataset com os textos integrais das decisões (ZIPs de TXTs)
DATASET_TEXTOS = 'integras-de-decisoes-terminativas-e-acordaos-do-diario-da-justica'

# ─── Arquivos ─────────────────────────────────────────────────────────────────
PARQUET_BASE  = 'espelhos_acordaos_artigo2026.parquet'
PARQUET_SAIDA = 'espelhos_acordaos_artigo2026_com_texto.parquet'

# Pasta local para cache dos ZIPs (evita re-download)
DOWNLOAD_DIR = Path('downloads_stj')
DOWNLOAD_DIR.mkdir(exist_ok=True)

# Coluna chave de junção entre parquet e ZIPs
COL_SEQ = 'seq_documento_acordao'

print(f'CKAN       : {CKAN_BASE_URL}')
print(f'Dataset    : {DATASET_TEXTOS}')
print(f'Parquet    : {PARQUET_BASE}')
print(f'Cache ZIPs : {DOWNLOAD_DIR.resolve()}')

CKAN       : https://dadosabertos.web.stj.jus.br
Dataset    : integras-de-decisoes-terminativas-e-acordaos-do-diario-da-justica
Parquet    : espelhos_acordaos_artigo2026.parquet
Cache ZIPs : /mnt/d/wsl_pucdev/agent-orchestration-2026/notebooks/downloads_stj


## 2. Carregamento do Parquet Base

In [2]:
df = pd.read_parquet(PARQUET_BASE)

print(f'Total de registros : {len(df)}')
print(f'Colunas            : {df.columns.tolist()}')
print()
df.head(3)

Total de registros : 1225
Colunas            : ['id_peca', 'seq_documento', 'seq_documento_acordao', 'dt_publicacao', 'dt_documento_stj', 'ano', 'num_registro', 'dataDecisao', 'id_espelho']



,id_peca,seq_documento,seq_documento_acordao,dt_publicacao,dt_documento_stj,ano,num_registro,dataDecisao,id_espelho
0,202202853462.67.,185167822,188798478,2023-06-01,2023-05-10,2023,202202853462,20230510,000844657
1,202201555326.20.,156216757,156649445,2022-06-20,2022-06-14,2022,202201555326,20220614,000818176
2,202201341093.25.,155311792,155998284,2022-06-15,2022-06-07,2022,202201341093,20220607,000817528


In [3]:
# Constrói o conjunto de seq_documento_acordao que precisamos carregar
assert COL_SEQ in df.columns, f'❌ Coluna "{COL_SEQ}" não encontrada no parquet!'

# Dicionário: seq_documento_acordao (int) → lista de índices do dataframe
seq_para_idx: dict[int, list[int]] = {}
for idx, val in df[COL_SEQ].dropna().items():
    seq = int(val)
    seq_para_idx.setdefault(seq, []).append(idx)

seqs_necessarios: set[int] = set(seq_para_idx.keys())

print(f'seq_documento_acordao únicos necessários: {len(seqs_necessarios)}')
print(f'Exemplos: {list(seqs_necessarios)[:5]}')

# Anos presentes no parquet — usados para filtrar ZIPs pelo prefixo do nome
anos_necessarios: set[str] = set()
if 'ano' in df.columns:
    anos_necessarios = {str(int(a)) for a in df['ano'].dropna().unique()}
print(f'Anos necessários: {sorted(anos_necessarios)}')


seq_documento_acordao únicos necessários: 1225
Exemplos: [180578308, 196177927, 207640583, 245497864, 212580362]
Anos necessários: ['2022', '2023', '2024', '2025']


## 3. Obtenção dos Recursos CKAN

Lista os ZIPs disponíveis no dataset de textos integrais —
**uma única chamada de API** em vez de uma por documento.

In [4]:
def obter_metadados_dataset(dataset_id: str) -> dict:
    """Retorna os metadados de um dataset do CKAN."""
    url = f'{CKAN_BASE_URL}/api/3/action/package_show'
    response = requests.get(url, params={'id': dataset_id}, timeout=30)
    response.raise_for_status()
    return response.json()['result']


print(f'Obtendo lista de recursos do dataset "{DATASET_TEXTOS}"...')
metadados = obter_metadados_dataset(DATASET_TEXTOS)

# Filtra apenas recursos ZIP
recursos_zip = [
    {'name': r['name'], 'url': r['url'], 'resource_id': r['id']}
    for r in metadados.get('resources', [])
    if r.get('name', '').lower().endswith('.zip') or r.get('format', '').upper() == 'ZIP'
]

print(f'Recursos ZIP encontrados: {len(recursos_zip)}')
for r in recursos_zip[:10]:
    print(f'  {r["name"]}')
if len(recursos_zip) > 10:
    print(f'  ... (+{len(recursos_zip)-10} mais)')

Obtendo lista de recursos do dataset "integras-de-decisoes-terminativas-e-acordaos-do-diario-da-justica"...
Recursos ZIP encontrados: 893
  202202.zip
  202203.zip
  202204.zip
  20220502.zip
  20220503.zip
  20220504.zip
  20220505.zip
  20220506.zip
  20220509.zip
  20220510.zip
  ... (+883 mais)


## 4. Extração dos Textos dos ZIPs

Cada ZIP é baixado com cache local (`downloads_stj/`).
O conteúdo é varrido em memória — apenas os TXTs com
`seq_documento_acordao` necessários são lidos.

In [7]:
def baixar_com_cache(url: str, nome_arquivo: str) -> Path:
    """Baixa o arquivo para DOWNLOAD_DIR se não existir. Retorna o caminho local."""
    caminho = DOWNLOAD_DIR / nome_arquivo
    if caminho.is_file():
        print(f'  [cache] {nome_arquivo:<40}', end='\r', flush=True)
        return caminho
    print(f'  [↓] {nome_arquivo:<40}', end='\r', flush=True)
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(caminho, 'wb') as f:
            for chunk in r.iter_content(chunk_size=65536):
                if chunk:
                    f.write(chunk)
    print(f'  [✓] {nome_arquivo:<40}', end='\r', flush=True)
    return caminho


def processar_zip(
    caminho_zip: Path,
    seqs_validos: set[int],
    textos: dict[int, str],
) -> int:
    """
    Varre o ZIP em memória.
    Para cada TXT cujo nome (stem) é um seq_documento_acordao no conjunto,
    lê o conteúdo e armazena em `textos[seq]`.
    Retorna o número de textos extraídos neste ZIP.
    """
    extraidos = 0
    with zipfile.ZipFile(caminho_zip, 'r') as zf:
        for membro in zf.namelist():
            if not membro.lower().endswith('.txt'):
                continue
            stem = Path(membro).stem
            try:
                seq = int(stem)
            except ValueError:
                continue
            if seq not in seqs_validos:
                continue
            # Lê o conteúdo do TXT (tenta UTF-8, fallback latin-1)
            raw = zf.read(membro)
            try:
                txt = raw.decode('utf-8')
            except UnicodeDecodeError:
                txt = raw.decode('latin-1')
            txt = txt.replace('<br>', '\n').strip()
            textos[seq] = txt
            extraidos += 1
    return extraidos


print('✅ Funções de extração definidas.')

✅ Funções de extração definidas.


In [ ]:
# Dicionário: seq_documento_acordao → texto
textos_extraidos: dict[int, str] = {}

# Conjunto ainda não encontrado (para parar cedo se já tivermos tudo)
seqs_pendentes = set(seqs_necessarios)

# Filtra apenas os ZIPs cujo nome começa com um dos anos de interesse
if anos_necessarios:
    recursos_filtrados = [
        r for r in recursos_zip
        if any(r['name'].startswith(ano) for ano in anos_necessarios)
    ]
else:
    recursos_filtrados = recursos_zip

print(f'Iniciando extração de {len(seqs_pendentes)} textos em {len(recursos_filtrados)} ZIPs'
      + (f' [anos: {sorted(anos_necessarios)}]' if anos_necessarios else ' [todos os anos]') + '...')
print('Caso ocorra erro em algum arquivo, pode-se tentar novamente. Os arquivos são baixados na pasta "./downloads_stj"')
print(f'  [✓] Extração concluída _o/')

for recurso in tqdm(recursos_filtrados, desc='ZIPs'):
    if not seqs_pendentes:
        print('\n✅ Todos os textos encontrados — interrompendo varredura antecipada.')
        break

    try:
        caminho = baixar_com_cache(recurso['url'], recurso['name'])
        n = processar_zip(caminho, seqs_pendentes, textos_extraidos)
        # Remove os encontrados do conjunto pendente
        seqs_pendentes -= set(textos_extraidos.keys())
        if n:
            print(f'  → {n} texto(s) extraído(s) | pendentes: {len(seqs_pendentes)}', end='\r', flush=True)
    except Exception as e:
        print(f'  ⚠️  Erro em {recurso["name"]}: {e}')

print()
print(f'Textos extraídos : {len(textos_extraidos)} / {len(seqs_necessarios)}')
print(f'Não encontrados  : {len(seqs_pendentes)}')
if seqs_pendentes:
    print(f'  Exemplos: {list(seqs_pendentes)[:5]}')

Iniciando extração de 1225 textos em 858 ZIPs [anos: ['2022', '2023', '2024', '2025']]...
Caso ocorra erro em algum arquivo, pode-se tentar novamente. Os arquivos são baixados na pasta "./downloads_stj"
  [✓] Extração concluída _o/


ZIPs:   0%|          | 0/858 [00:00<?, ?it/s]

  ⚠️  Erro em 20220505.zip: 522 Server Error: <none> for url: https://dadosabertos.web.stj.jus.br/dataset/a2cd85cc-1391-4ebc-aeed-a45dd75a7987/resource/35036698-c7f0-491d-be29-46c2ce641294/download/20220505.zip
  ⚠️  Erro em 20220531.zip: 522 Server Error: <none> for url: https://dadosabertos.web.stj.jus.br/dataset/a2cd85cc-1391-4ebc-aeed-a45dd75a7987/resource/0eb2b208-37bd-416e-a5b1-4e7fabdd00a3/download/20220531.zip
  ⚠️  Erro em 20220603.zip: 522 Server Error: <none> for url: https://dadosabertos.web.stj.jus.br/dataset/a2cd85cc-1391-4ebc-aeed-a45dd75a7987/resource/99976e85-04ce-4e8e-9852-87d5f914d5ae/download/20220603.zip
  ⚠️  Erro em 20220608.zip: 522 Server Error: <none> for url: https://dadosabertos.web.stj.jus.br/dataset/a2cd85cc-1391-4ebc-aeed-a45dd75a7987/resource/2eebd134-4a96-41ea-a24a-dca292a34430/download/20220608.zip
  ⚠️  Erro em 20220617.zip: 522 Server Error: <none> for url: https://dadosabertos.web.stj.jus.br/dataset/a2cd85cc-1391-4ebc-aeed-a45dd75a7987/resource/fdaa

## 5. Junção dos Textos ao Dataframe

In [ ]:
df_resultado = df.copy()

# Mapeia texto via seq_documento_acordao
df_resultado['texto'] = df_resultado[COL_SEQ].apply(
    lambda seq: textos_extraidos.get(int(seq), '') if pd.notna(seq) else ''
)

# Estatísticas
com_texto = df_resultado['texto'].str.len().gt(0).sum()
sem_texto = len(df_resultado) - com_texto

print('=== RESULTADO ===')
print(f'Total de registros : {len(df_resultado)}')
print(f'Com texto          : {com_texto}')
print(f'Sem texto          : {sem_texto}')
print()
print('Tamanho médio do texto (chars): ',
      df_resultado.loc[df_resultado['texto'].str.len()>0, 'texto'].str.len().mean().round(0))

## 6. Preview

In [ ]:
# Exibe um exemplo de texto extraído
com_texto_df = df_resultado[df_resultado['texto'].str.len() > 0]

if not com_texto_df.empty:
    row = com_texto_df.iloc[0]
    print(f'seq_documento_acordao : {row[COL_SEQ]}')
    if 'id_espelho' in row:
        print(f'id_espelho            : {row["id_espelho"]}')
    if 'nomeOrgaoJulgador' in row:
        print(f'Órgão julgador        : {row.get("nomeOrgaoJulgador", "")}')
    print()
    print('=== TEXTO (primeiros 3000 chars) ===')
    print(row['texto'][:3000])
else:
    print('⚠️  Nenhum texto encontrado ainda. Verifique os ZIPs disponíveis no CKAN.')

## 7. Salvamento do Resultado

In [ ]:
df_resultado.to_parquet(PARQUET_SAIDA, index=False)

print(f'✅ Arquivo salvo: {PARQUET_SAIDA}')
print(f'   Registros : {len(df_resultado)}')
print(f'   Colunas   : {df_resultado.columns.tolist()}')
print(f'   Tamanho   : {Path(PARQUET_SAIDA).stat().st_size / 1024:.1f} KB')